# Goal and functionality of algorithm
Initiate organisms and watch them respond to stimuli through time. After a fixed number of time-steps, a reproduction condition triggers. Upon reproduction, the organisms which meet reproduction condition(s) reproduce (with a chance for mutation), all of the previous generation dies. This sequence repeats indefinitely. 


# Steps for a single generation
1. Initiate world (including state of n organisms) at t = 0
2. Allow organisms to perceive their situation and make a decision as to what to do. Taking action if applicable
3. Repeat step 2 for all timesteps

# Inter-generational steps
1. Run a single generation
2. Evaluate and execute reproduction condition. Reproduction should enable both passing of genetic information as well as the addition of new genetic information through mutation.
3. Repeat step 2 for n generations

# What does the MVP look like?
- organisms have a small number of neurons that map to some perception/action workflow
    - does this always need to look like [perception] -> [action]
- organisms can update state based on some perception of the world
- a population of organisms can reproduce based on some condition
- organisms can evolve (including passing genes and random mutations)

---
# Scratchpad

The MVP above is built and then some. `uv run python execute.py` runs it with the animation; the cells below are for poking at individual creatures and working out *why* a behaviour appeared.

The answer to "does this always need to look like [perception] -> [action]" turned out to be no: inner neurons keep their value between timesteps, so a genome can wire perception -> memory -> action, or a loop that ignores perception entirely.

Start the kernel with `uv run jupyter lab` so the notebook picks up the project environment.

In [ ]:
%matplotlib inline
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

import inspect_utils
from capability_utils import Action
from organism import World
from settings import Settings

config = Settings()

In [ ]:
def draw_world(world, ax, title):
    """Zones, scent, barriers and creatures, in the order they should stack."""
    extent = (0, world.width, 0, world.height)
    for shading in world.objective.zones(world):
        ax.imshow(
            shading.mask.T,
            origin="lower",
            cmap=shading.colour,
            alpha=0.3,
            extent=extent,
            vmin=0,
            vmax=1,
        )
    ax.imshow(
        world.pheromone.T.clip(0, 1),
        origin="lower",
        cmap="BuPu",
        alpha=0.55,
        extent=extent,
        vmin=0,
        vmax=1,
    )
    ax.imshow(
        np.ma.masked_where(~world.barriers.T, world.barriers.T),
        origin="lower",
        cmap="Greys",
        alpha=0.85,
        extent=extent,
        vmin=0,
        vmax=1,
    )
    xs, ys = world.positions()
    ax.scatter(xs, ys, s=7)
    ax.set_title(title)
    ax.set_xlim(0, world.width)
    ax.set_ylim(0, world.height)

## One organism, up close

In [ ]:
world = World(config=config)
org = world.organisms[0]

# Everything it can sense right now.
{str(sensor): round(value, 3) for sensor, value in org.perceive(world).items()}

In [ ]:
# Its whole brain, one connection per line.
print(org.brain.describe())
print()
print("senses it actually consults:", [str(s) for s in org.brain.needed_sensors])

In [ ]:
# The same brain as a picture: blue excites, orange inhibits,
# and thickness follows the weight.
inspect_utils.draw_brain(org.brain, config);

In [ ]:
# What its action neurons want to do this timestep.
levels = org.brain.think(org, world)
{str(action): round(levels[action], 3) for action in Action}

## Watching a population evolve

In [ ]:
world = World(config=config, objective="left")

history = [world.run_generation() / world.n_organisms for _ in range(40)]

plt.plot(history)
plt.xlabel("generation")
plt.ylabel("fraction surviving")
plt.ylim(0, 1);

In [ ]:
# Where an evolved generation ends up, versus where it started.
figure, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharex=True, sharey=True)
draw_world(world, axes[0], "start of generation")
world.run_generation()
draw_world(world, axes[1], "end of generation")

## Obstacles and scent

Barriers turn "head west" from a complete solution into one that strands a creature in a dead end. Pheromone is the only channel one creature has for changing what another perceives — look for trails in the purple layer.

In [ ]:
hard = replace(config, barrier_layout="slalom")
maze = World(config=hard, objective="left")
for _ in range(25):
    maze.run_generation()

figure, ax = plt.subplots(figsize=(6.5, 6.5))
maze.run_generation()
draw_world(maze, ax, "after 25 generations against the slalom")

## Racing objectives against each other

In [ ]:
quick = replace(config, n_organisms=150, steps_per_generation=120)

for name in ["left", "corners", "stay", "hazard"]:
    trial = World(config=quick, objective=name)
    curve = [trial.run_generation() / quick.n_organisms for _ in range(25)]
    plt.plot(curve, label=name)

plt.xlabel("generation")
plt.ylabel("fraction surviving")
plt.ylim(0, 1)
plt.legend();

## Changing the rules without touching the code

`Settings` is frozen, so derive a variant with `replace` rather than editing globals.

In [ ]:
# How much does the mutation rate matter?
for rate in [0.005, 0.02, 0.08]:
    tuned = replace(config, point_mutation_rate=rate, n_organisms=150, steps_per_generation=120)
    trial = World(config=tuned, objective="corners")
    curve = [trial.run_generation() / tuned.n_organisms for _ in range(25)]
    plt.plot(curve, label=f"mutation rate {rate}")

plt.xlabel("generation")
plt.ylabel("fraction surviving")
plt.ylim(0, 1)
plt.legend();

## Keeping a creature you like

In [ ]:
best = world.organisms[0]
inspect_utils.save_genome(best.genome, config, "good_creature.json")

restored = inspect_utils.load_genome("good_creature.json")
print("round-trips exactly:", restored == best.genome)

## Ideas to try

- A new objective in `objectives.py` — the animation draws its zones automatically.
- A new member of `Sensor` or `Action` in `capability_utils.py` — evolution picks it up on the next run with no other changes.
- Crossover between two survivors, instead of the current asexual mutation-only reproduction.
- Partial credit in selection, so hard objectives have a gradient to climb instead of reseeding from scratch whenever nobody survives.